In [ ]:
import subprocess
import time
import requests
from datetime import datetime

def check_ollama_server(base_url="http://localhost:11434"):
    """Check if Ollama server is responding"""
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False

def start_ollama_server(max_retries=3, wait_time=10):
    """Start Ollama server and wait for it to be ready"""
    print("Starting Ollama server...")
    
    try:
        # Start the server in the background
        process = subprocess.Popen(
            ["ollama", "serve"], 
            stdout=subprocess.PIPE, 
            stderr=subprocess.PIPE,
            text=True
        )
        
        # Wait for server to start up
        for attempt in range(max_retries):
            time.sleep(wait_time)
            if check_ollama_server():
                print(f"Ollama server started successfully (attempt {attempt + 1})")
                return process
            print(f"Server not ready yet, waiting... (attempt {attempt + 1}/{max_retries})")
        
        # If we get here, server failed to start
        print("Failed to start Ollama server after maximum retries")
        process.terminate()
        return None
        
    except Exception as e:
        print(f"Error starting Ollama server: {e}")
        return None

def ensure_ollama_server_running():
    """Ensure Ollama server is running, restart if necessary"""
    if check_ollama_server():
        print("Ollama server is running")
        return True
    
    print("Ollama server is not responding, attempting to restart...")
    server_process = start_ollama_server()
    return server_process is not None

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Ensure Ollama server is running before starting
if not ensure_ollama_server_running():
    print("Failed to start Ollama server. Exiting...")
    exit(1)

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Check if server is still running before processing
        if not check_ollama_server():
            print("Ollama server stopped responding, attempting to restart...")
            with open(progress_file, "a") as file:
                file.write(f"[{current_time}] Server failure detected, restarting...\n")
            
            if not ensure_ollama_server_running():
                raise Exception("Could not restart Ollama server")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench-a"
        output_dir = "output/sysengbench-a/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)
        
        # Log successful completion
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Successfully processed model: {model}\n")

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        
        # Check if it's a server-related error and try to restart
        if "connection" in str(e).lower() or "server" in str(e).lower():
            print("Detected potential server issue, attempting restart...")
            with open(progress_file, "a") as file:
                file.write(f"[{current_time}] Server error detected: {e}\n")
                file.write(f"[{current_time}] Attempting server restart...\n")
            
            if ensure_ollama_server_running():
                print("Server restarted successfully, retrying model...")
                # Retry the current model once
                try:
                    subprocess.run(command, shell=True, check=True)
                    with open(progress_file, "a") as file:
                        file.write(f"[{current_time}] Successfully processed model {model} after server restart\n")
                    continue  # Skip adding to failed_models
                except Exception as retry_error:
                    print(f"Retry failed for model {model}: {retry_error}")
                    with open(progress_file, "a") as file:
                        file.write(f"[{current_time}] Retry failed for model {model}: {retry_error}\n")
        
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)

# Final summary
print(f"\n--- Batch Processing Complete ---")
print(f"Total models processed: {total_models}")
print(f"Failed models: {len(failed_models)}")
if failed_models:
    print(f"Failed models list: {failed_models}")

# Log the final summary
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
with open(progress_file, "a") as file:
    file.write(f"\n[{current_time}] Batch Processing Complete\n")
    file.write(f"Total models: {total_models}\n")
    file.write(f"Failed models: {len(failed_models)}\n")
    if failed_models:
        file.write(f"Failed models list: {', '.join(failed_models)}\n")